In [11]:
from google.colab import drive
drive.mount('COLAB_DRIVE')

Drive already mounted at COLAB_DRIVE; to attempt to forcibly remount, call drive.mount("COLAB_DRIVE", force_remount=True).


In [12]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import tiktoken

import numpy as np

from beartype import beartype
import json
import glob

def is_colab() -> bool:
    try:
        import google.colab  # type: ignore
        return True
    except Exception:
        return False

project_name = "nanoGPT" # variable depends on project

if is_colab():
    matches = glob.glob(f"COLAB_DRIVE/MyDrive/**/{project_name}/root_path.py", recursive=True)
    print(matches)
    if len(matches) == 1:
        proj_dir = os.path.dirname(matches[0])
    with open(os.path.join(proj_dir, "config.json"), "r") as f:
        config = json.load(f)
    config['seed_settings']['use_seed'] = False

else:
    from root_path import get_root_path
    proj_dir = get_root_path()
    print("Is local")
    with open(os.path.join(proj_dir, "config.json"), "r") as f:
        config = json.load(f)

import sys
sys.path.append(proj_dir)

from utils import torch_ckpt
from model import model

['COLAB_DRIVE/MyDrive/Project/nanoGPT/root_path.py']


In [13]:
file_path = os.path.join(proj_dir, "Trainer", "Trainer.ipynb")

config['path_settings']['file_path'] = file_path
config['git_settings']['strict_git'] = False

device = config['env_settings']['device']
max_iters = config['deep_learning_settings']['trainer_config']['max_iters']

## Seed Setting

In [14]:
seed_test = torch_ckpt.ckpt_manager(**config)

use_deterministic == True
[Warning] [Git] Not a git repository. Run 'git init' to enable git tracking.


## Data Preparation

In [15]:
sys.path

['/content',
 '/env/python',
 '/usr/lib/python312.zip',
 '/usr/lib/python3.12',
 '/usr/lib/python3.12/lib-dynload',
 '',
 '/usr/local/lib/python3.12/dist-packages',
 '/usr/lib/python3/dist-packages',
 '/usr/local/lib/python3.12/dist-packages/IPython/extensions',
 '/root/.ipython',
 'COLAB_DRIVE/MyDrive/Project/nanoGPT',
 '/tmp/tmp7ta5isen',
 'COLAB_DRIVE/MyDrive/Project/nanoGPT']

In [16]:
from data.data_loader import TokenDataLoader

data_loader = TokenDataLoader(data_train_path=os.path.join(proj_dir, "data", "tokenized", "concatenated", "train_all.bin"),
                              data_val_path=os.path.join(proj_dir, "data", "tokenized", "concatenated", "val_all_10_to_11.bin"),
                              token_len=config['deep_learning_settings']['model_config']['token_len'],
                              batch_size=config['deep_learning_settings']['data_config']['batch_size'])



## Model definition

In [17]:
if "ff_dim" in config['deep_learning_settings']['model_config'].keys():
    config['deep_learning_settings']['model_config'].pop('ff_dim', None)

In [18]:
gpt_model = model.GPT(**config['deep_learning_settings']['model_config'])

## Optimizer

In [19]:
criterion = nn.CrossEntropyLoss()  # 예: 분류 문제일 때
optimizer = optim.Adam(gpt_model.parameters(), lr=config['deep_learning_settings']['optimizer_config']['lr'])

## Data Batch Module 화

In [20]:
from root_path import get_root_path
import random
import torch
import numpy as np

print("="*50)
print("trainer start:")
print("device:", device)
print("="*50)

train_loss_list = {}
val_loss_list = {}

train_loss_acc = []
val_loss_acc = []

train_loss_per_iter_eval = {}
val_loss_per_iter_eval = {}

gpt_model.to(device)

for _idx in range(max_iters):
    print("="*10)
    print(f"Iters: {_idx+1}/{max_iters}")
    x,y = data_loader.get_train_batch()
    x,y = x.to(device), y.to(device)
    pred = gpt_model(x)
    B, T, C = pred.size()
    train_loss = criterion(pred.view(B*T, C), y.view(B*T))
    train_loss.backward()
    optimizer.step()
    optimizer.zero_grad()

    train_loss_acc.append({"batch_size":B, "train_loss_sum":train_loss.item() * B})

    with torch.no_grad():
        x,y = data_loader.get_val_batch()
        x,y = x.to(device), y.to(device)
        gpt_model.to(device)
        pred = gpt_model(x)
        B, T, C = pred.size()
        val_loss = criterion(pred.view(B*T, C), y.view(B*T))
    val_loss_acc.append({"batch_size":B, "val_loss_sum":val_loss.item() * B})

    if _idx % config['deep_learning_settings']['trainer_config']['iter_eval'] == 0:
        sum_of_train_loss = sum(list(map(lambda x: x['train_loss_sum'], train_loss_acc)))
        sum_of_batch_size = sum(list(map(lambda x: x['batch_size'], train_loss_acc)))
        train_loss_per_iter_eval = sum_of_train_loss / sum_of_batch_size
        train_loss_list[_idx] = train_loss_per_iter_eval
        train_loss_acc = []
        print(f"{_idx}th iter train loss:{train_loss_per_iter_eval:.3f}")


        val_loss_per_iter_eval = sum(list(map(lambda x: x['val_loss_sum'], val_loss_acc))) / sum(list(map(lambda x: x['batch_size'], val_loss_acc)))
        val_loss_list[_idx] = val_loss_per_iter_eval
        val_loss_acc = []
        print(f"{_idx}th iter val loss:{val_loss_per_iter_eval:.3f}")

trainer start:
device: cuda
Iters: 1/100000000
0th iter train loss:11.368
0th iter val loss:9.972
Iters: 2/100000000
Iters: 3/100000000
Iters: 4/100000000
Iters: 5/100000000
Iters: 6/100000000
Iters: 7/100000000
Iters: 8/100000000
Iters: 9/100000000
Iters: 10/100000000
Iters: 11/100000000
Iters: 12/100000000
Iters: 13/100000000
Iters: 14/100000000
Iters: 15/100000000
Iters: 16/100000000
Iters: 17/100000000
Iters: 18/100000000
Iters: 19/100000000
Iters: 20/100000000
Iters: 21/100000000
Iters: 22/100000000
Iters: 23/100000000
Iters: 24/100000000
Iters: 25/100000000
Iters: 26/100000000
Iters: 27/100000000
Iters: 28/100000000
Iters: 29/100000000
Iters: 30/100000000
Iters: 31/100000000
Iters: 32/100000000
Iters: 33/100000000
Iters: 34/100000000
Iters: 35/100000000
Iters: 36/100000000
Iters: 37/100000000
Iters: 38/100000000
Iters: 39/100000000
Iters: 40/100000000
Iters: 41/100000000
Iters: 42/100000000
Iters: 43/100000000
Iters: 44/100000000
Iters: 45/100000000
Iters: 46/100000000
Iters: 47/

KeyboardInterrupt: 

### 2025-12-29에 실행한 결과, 5분에 100 iter이다.
### - 500 iter까지 실행한 결과, 약 15분 걸렸다.
